In [1]:
import math
class Value:
    def __init__(self,data,children = (),_op = ""):
        self.data= data
        self._prev = set(children)
        self.grad = 0
        self._backward = lambda : None
        self._op = _op
    
    def __add__(self,other):
        other = other if isinstance(other,Value) else Value(other)
        out = Value(self.data + other.data,(self,other),"+")
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __radd__(self,other):
        out = self + other
        return out

    def __sub__(self,other):
        other = other if isinstance(other,Value) else Value(other)
        out = self + other*(-1)
        return out

    def __rsub__(self,other):
        out = -1*self + other
        return out

    def __mul__(self,other):
        other = other if isinstance(other,Value) else Value(other)
        out = Value(self.data * other.data,(self,other),"*")
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __rmul__(self,other):
        out = self*other
        return out

    def __pow__(self,other):
        assert isinstance(other,(int,float)), "only supporting int or float powers for now"
        out = Value(self.data**other,(self,),f"**{other}")
        def _backward():
            self.grad += (other*self.data**(other-1))* out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t,(self,),"tanh")
        def _backward():
            self.grad += (1-t**2)* out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        #recursive implementation of topological sort for backpropagation
        def build_topo(self):
            if self not in visited:
                visited.add(self)
                for child in self._prev:
                    build_topo(child)
                topo.append(self)
        build_topo(self)
        #initialising the gradient of the first parent node to 1, as it is set by default to 0
        self.grad = 1.0
        #execution of backpropagation
        for node in reversed(topo):
            node._backward()

    def __repr__(self):
        return f"Value(data={self.data})"



In [2]:
import random
# x: list
# nin: int
# nout: int
# nouts: list

class Neuron:
    def __init__(self,nin):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,1))

    def __call__(self,x):
        act = sum((wi*xi for wi,xi in zip(self.w,x)),self.b)
        out = act.tanh()
        return out

    def parameters(self):
        out = self.w +[self.b]
        return out

class Layer:
    def __init__(self,nin,nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self,x):
        outs =  [neuron(x) for neuron in self.neurons]
        return outs if len(outs)>1 else outs[0]

    def parameters(self):
        out = [parameter for neuron in self.neurons for parameter in neuron.parameters()]
        return out

class MLP:
    def __init__(self,nin,nouts):
        sizes = [nin] + nouts
        self.layers = [Layer(sizes[i],sizes[i+1]) for i in range(len(nouts))]

    def __call__(self,x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        out = [parameter for layer in self.layers for neuron in layer.neurons for parameter in neuron.parameters()]
        return out 



In [3]:
#training set
xs = [[2.0,3.5,-2.0],[4.0,0.5,1.5],[-1.0,-1.5,1.0],[2.5,1.7,-0.3]]
ys = [1.0,-1.0,1.0,-1.0] 

In [4]:
#Initialization of weights and biases
n = MLP(3,[4,4,1])

In [5]:
#forward pass
y_pred = [n(x) for x in xs]
loss = sum((y_real - y_out)**2 for y_real,y_out in zip(ys,y_pred))
print(f"Loss = {loss}")

Loss = Value(data=2.7528101069355113)


In [6]:
#backpropagation
for p in n.parameters():
    p.grad = 0
loss.backward()

In [7]:
#update
for p in n.parameters():
    p.data += -0.01*p.grad

In [8]:
def training(k,step_size):
    for i in range(k):
        #forward pass
        y_pred = [n(x) for x in xs]
        loss = sum((y_real - y_out)**2 for y_real,y_out in zip(ys,y_pred))
        if (i+1) % 10 == 0: 
            print(f"step {i+1}: Loss = {loss}")

        #backpropagation
        for p in n.parameters():
            p.grad = 0
        loss.backward()

        #update
        for p in n.parameters():
            p.data += -step_size*p.grad

In [9]:
training(100,0.05)

step 10: Loss = Value(data=0.09966128476147709)
step 20: Loss = Value(data=0.04092203898534421)
step 30: Loss = Value(data=0.02508012840193947)
step 40: Loss = Value(data=0.017857855392452146)
step 50: Loss = Value(data=0.013771139857435416)
step 60: Loss = Value(data=0.011159328144307293)
step 70: Loss = Value(data=0.009353658646314017)
step 80: Loss = Value(data=0.008034664418703534)
step 90: Loss = Value(data=0.007031097457364208)
step 100: Loss = Value(data=0.006243168815935918)


In [10]:
#results on our training set
y_pred = [n(x) for x in xs]
y_pred

[Value(data=0.9453919085635715),
 Value(data=-0.9765933936478166),
 Value(data=0.9708731126089399),
 Value(data=-0.9576285565121296)]